# Leakage: cómo te engañas sin querer (y crees que tu modelo es genial)

**Facsímil 8 · La ciencia de los datos** — capítulos 2 y 3
(calidad de datos y *leakage*; *splits*, muestreo y medir sin engañarse).

Casi todos los «modelos increíbles» que fracasan al desplegarse tenían **fuga de datos** (*leakage*):
información del test —o del futuro— que se coló en el entrenamiento. Es el error más traicionero del
*machine learning* porque **no da ningún error**: da *buenas noticias falsas*. En este cuaderno lo
provocas a propósito, ves un modelo presumir de un acierto altísimo sobre datos donde **no hay nada que
aprender**, descubres de dónde sale el truco, y lo arreglas para ver el número real, el del puro azar.
Aprender a oler la fuga vale más que cualquier algoritmo.

### Qué vas a aprender
- Qué es el *leakage* y por qué es silencioso (no rompe nada, infla resultados).
- A provocar el caso canónico: **seleccionar features mirando todo el dataset** antes de partir.
- A hacerlo **bien**: separar el test antes de tocar nada y meter cada decisión dentro de la validación.
- Otros tipos de fuga (target leakage, escalado, temporal) y cómo crece el problema con más columnas.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
np.random.seed(0)

# 120 muestras, 1000 columnas de RUIDO puro, y una etiqueta ALEATORIA: no hay
# ninguna relacion entre X e y. Si un modelo "acierta" aqui, es trampa segura.
X = np.random.normal(size=(120, 1000))
y = np.random.randint(0, 2, 120)
print("Datos:", X.shape, "-> 1000 columnas de ruido y una etiqueta al azar. No hay senal que aprender.")
print("Cualquier acierto por encima del 50% (azar) sera, por definicion, una ilusion.")


Datos: (120, 1000) -> 1000 columnas de ruido y una etiqueta al azar. No hay senal que aprender.
Cualquier acierto por encima del 50% (azar) sera, por definicion, una ilusion.


## 1. El escenario más honesto posible

Para que no haya dudas, hemos construido un caso donde **sabemos** que no hay nada que aprender: la
etiqueta es una moneda al aire, independiente de las 1000 columnas de ruido. El mejor modelo posible
aquí debería acertar el 50% (azar puro). Si conseguimos que «acierte» más, será 100% trampa, y podremos
señalar exactamente de dónde viene. Es el experimento clásico de Ambroise y McLachlan que destapó este
error en miles de estudios.


## 2. La trampa: elegir las features mirando *todos* los datos

Con 1000 columnas de ruido, algunas parecerán predictivas **por puro azar** (siempre que miras
suficientes cosas al azar, unas pocas correlacionan por casualidad). El error clásico: seleccionar las
«mejores» features usando el dataset **entero** y *después* partir en train/test. El test ya ha influido
en la elección de las columnas: hemos hecho trampa sin darnos cuenta.


In [2]:
# MAL: seleccionamos las 20 mejores features con TODO el dataset (incluido el test)
X_filtrado = SelectKBest(f_classif, k=20).fit_transform(X, y)
acc_tramposa = cross_val_score(LogisticRegression(max_iter=1000), X_filtrado, y, cv=5).mean()
print(f"Acierto (con fuga):  {acc_tramposa:.3f}   <- 'parece un modelo estupendo' sobre RUIDO")


Acierto (con fuga):  0.775   <- 'parece un modelo estupendo' sobre RUIDO


## 3. Hacerlo bien: el test no se toca hasta el final

La regla de oro: **cualquier** decisión que mire los datos (escalar, seleccionar features, imputar
valores) se aprende **solo con el train**. Metemos la selección de features *dentro* del pipeline de
validación cruzada, para que en cada pliegue el test esté de verdad escondido cuando se eligen las
columnas.


In [3]:
# BIEN: la seleccion va DENTRO del pipeline; se reaprende solo con el train de cada pliegue
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
acc_honesta = cross_val_score(pipe, X, y, cv=5).mean()
print(f"Acierto (con fuga):    {acc_tramposa:.3f}")
print(f"Acierto (honesto):     {acc_honesta:.3f}   <- ~0.50, el del azar: NO hay nada que aprender")
print(f"\nLa fuga inflaba el resultado en {100*(acc_tramposa-acc_honesta):.0f} puntos, sobre datos sin senal.")
print("Mismo modelo, mismos datos. La unica diferencia: CUANDO miramos el test.")


Acierto (con fuga):    0.775
Acierto (honesto):     0.450   <- ~0.50, el del azar: NO hay nada que aprender

La fuga inflaba el resultado en 32 puntos, sobre datos sin senal.
Mismo modelo, mismos datos. La unica diferencia: CUANDO miramos el test.


**Eso es leakage.** Ninguna línea daba error. El primer número era una **buena noticia falsa** que te
habría llevado a desplegar con confianza un modelo que en realidad no aprende nada. La fuga no se caza
con un *try/except*: se caza con disciplina, separando el test *antes* de tocar nada.


## 4. Experimento: cuanto más miras, más te engañas

La fuga por selección de features empeora cuanto **más columnas** tienes, porque con más candidatas es
más fácil encontrar correlaciones espurias por azar. Lo medimos: repetimos la trampa con 200, 500 y
2000 columnas de ruido y vemos cómo crece el acierto falso (recuerda: el honesto siempre es ~0,50).


In [4]:
print("nº de columnas de ruido | acierto CON fuga | acierto honesto")
for n_cols in [200, 500, 2000]:
    Xn = np.random.normal(size=(120, n_cols)); yn = np.random.randint(0, 2, 120)
    tramposa = cross_val_score(LogisticRegression(max_iter=1000),
                               SelectKBest(f_classif, k=20).fit_transform(Xn, yn), yn, cv=5).mean()
    honesta = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)),
                              Xn, yn, cv=5).mean()
    print(f"        {n_cols:>5}          |      {tramposa:.3f}      |     {honesta:.3f}")
print("\nMas columnas -> mas correlaciones espurias -> mayor acierto FALSO. El honesto se queda en el azar.")


nº de columnas de ruido | acierto CON fuga | acierto honesto
          200          |      0.717      |     0.550
          500          |      0.783      |     0.567
         2000          |      0.858      |     0.558

Mas columnas -> mas correlaciones espurias -> mayor acierto FALSO. El honesto se queda en el azar.


## 5. Otros disfraces de la fuga

La selección de features es solo un caso. La fuga tiene muchas caras, todas con el mismo pecado de fondo
(dejar que el test o el futuro influyan en el entrenamiento):

- **Target leakage:** una columna que contiene, directa o indirectamente, la respuesta. Por ejemplo,
  «importe ya devuelto» para predecir si un cliente devolverá: el modelo «acierta» porque le estás dando
  la solución.
- **Fuga por escalado/imputación:** calcular la media para normalizar usando *todo* el dataset antes de
  partir. Sutil, mismo pecado.
- **Fuga temporal:** usar datos del futuro para predecir el pasado (mezclar fechas al hacer el split en
  series temporales).

Todas se evitan igual: **separa el test primero** y aprende cada transformación solo con el train.


## 6. Pruébalo tú

1. **Target leakage:** añade una columna que sea `y + un poco de ruido`. El modelo «acierta» casísimo...
   porque le estás dando la respuesta. ¿La detectarías en un dataset real, donde no sabes que esa columna
   es tramposa?
2. **Fuga por escalado:** ajusta un `StandardScaler` sobre todo el dataset antes de partir y mide el
   optimismo. Más sutil, mismo efecto.
3. **Sube `k`** (selecciona más features): ¿crece la fuga? Cuantas más eliges mirando el test, peor.
4. **Repite varias veces** con distintas semillas: el acierto honesto baila alrededor de 0,50; el
   tramposo, siempre por encima.


## 7. Errores comunes

- **Preprocesar antes de partir.** Escalar, seleccionar o imputar con todo el dataset filtra información
  del test. Hazlo dentro de la validación.
- **Un resultado sospechosamente bueno y no desconfiar.** La primera pregunta no es «¿qué algoritmo?»,
  sino «¿se me ha colado el futuro o el test?».
- **Mirar el test muchas veces** para ajustar decisiones: acabas sobreajustando al test. Se mira una vez,
  al final.
- **Confiar en columnas «mágicas».** Una feature demasiado predictiva suele ser *target leakage*
  disfrazado.


## 8. Qué te llevas

- El **leakage** es información del test (o del futuro) que se cuela en el entrenamiento. No da error: da
  resultados demasiado buenos para ser verdad.
- La defensa es **separar el test antes de todo** y meter cada decisión (escalar, seleccionar, imputar)
  **dentro** de la validación.
- El problema **crece con el número de columnas** y tiene muchos disfraces (selección, target, escalado,
  temporal); todos se evitan con la misma disciplina.

**Para seguir:** el capítulo 3 profundiza en *splits* honestos; el siguiente cuaderno muestra otra forma
de engañarte: mirar solo el promedio y no los subgrupos.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*